# 티스토리 글 수집

원본 `python/tistory_crawling.ipynb`의 Selenium 검색·본문 수집 흐름을 공개용으로 정리한 코드입니다. 원본의 `datetime.now()`는 게시일이 아닌 수집일이므로, 여기서는 게시일 메타데이터를 확인할 수 없으면 `date`를 비워 둡니다. 검색 결과의 연도 범위가 게시일을 보증하지는 않습니다. 수집한 원문 CSV는 공개 저장소 밖에 보관합니다.


In [ ]:
import os
import re
import time
from pathlib import Path
from urllib.parse import quote_plus, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service


In [ ]:
# 게시일 추출: 확인되지 않은 날짜는 빈 값으로 유지
def get_post_date(content_soup):
    for key, value in [('property', 'article:published_time'),
                       ('property', 'og:regDate'),
                       ('name', 'date'),
                       ('name', 'pubdate')]:
        tag = content_soup.find('meta', attrs={key: value})
        if tag and tag.get('content'):
            date = pd.to_datetime(tag['content'], errors='coerce')
            if pd.notna(date):
                return date.strftime('%Y-%m-%d')

    tag = content_soup.find('time', attrs={'datetime': True})
    if tag:
        date = pd.to_datetime(tag['datetime'], errors='coerce')
        if pd.notna(date):
            return date.strftime('%Y-%m-%d')
    return None


def google_search_crawler(search_query, chrome_driver_path, start_year, end_year):
    service = Service(chrome_driver_path)
    driver = webdriver.Chrome(service=service)
    results = []

    try:
        for year in range(start_year, end_year + 1):
            base_url = (f'https://www.google.com/search?q={quote_plus(search_query)}'
                        f'&tbs=cdr:1,cd_min:1/1/{year},cd_max:12/31/{year}')
            driver.get(base_url)
            driver.implicitly_wait(10)

            while True:
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                search_results = soup.select('.tF2Cxc')

                for result in search_results:
                    try:
                        url = result.select_one('.yuRUbf a')['href']
                        if not urlparse(url).hostname.endswith('.tistory.com'):
                            continue
                        title = result.select_one('.LC20lb').get_text(strip=True)
                        response = requests.get(url, timeout=15)
                        response.raise_for_status()
                        response.encoding = response.apparent_encoding
                        content_soup = BeautifulSoup(response.text, 'html.parser')
                        paragraphs = content_soup.find_all('p')
                        content = '\n'.join([p.get_text(strip=True) for p in paragraphs
                                             if len(p.get_text().strip()) > 20])
                        content = re.sub(r'http[s]?://\S+', '', content)
                        date = get_post_date(content_soup)
                        results.append({'url': url, 'title': title,
                                        'content': content, 'date': date})
                    except Exception as e:
                        print(f'Error processing result: {e}')

                try:
                    next_button = driver.find_element(By.LINK_TEXT, '다음')
                    next_button.click()
                    time.sleep(2)
                except Exception:
                    print(f'{year}년 검색 크롤링 완료.')
                    break
    finally:
        driver.quit()

    return pd.DataFrame(results, columns=['url', 'title', 'content', 'date'])


In [ ]:
# 크롬드라이버와 비공개 데이터 폴더는 실행 환경에서 지정
chrome_driver_path = os.environ['CHROMEDRIVER_PATH']
data_path = Path(os.environ['LG_VOC_PRIVATE_DATA'])
output_folder = data_path / 'tistory'
output_folder.mkdir(parents=True, exist_ok=True)

# 필요한 검색어별로 실행; 원문 CSV는 공개 저장소에 넣지 않음
search_query = '의류관리기 site:tistory.com'
start_year = 2022
end_year = 2024
result = google_search_crawler(search_query, chrome_driver_path, start_year, end_year)
result = result.drop_duplicates(subset='url').reset_index(drop=True)
result.to_csv(output_folder / 'tistory_styler_script.csv', index=False, encoding='utf-8-sig')
print(f'수집 {len(result)}건, 게시일 확인 {result.date.notna().sum()}건')
